In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Canonical, Iterative, and Bayesian Quantum Phase Estimation with CUDA-Q — Solutions
$\renewcommand{\ket}[1]{|#1\rangle}\renewcommand{\bra}[1]{\langle#1|}$

Quantum Phase Estimation (QPE) is a foundational quantum algorithm that offers a significant theoretical advantage for simulating chemical systems. While the Variational Quantum Eigensolver (VQE) is suited for near-term noisy devices, QPE sits at the other end of the spectrum and likely requires a fault-tolerant quantum computer for practical applications. This notebook will guide you through developing an implementation of canonical QPE and help you explore two variants designed to overcome some of the resource constraints of the original version.

---

**What You Will Do:**
* Implement the Quantum Fourier Transform (QFT) and its inverse using CUDA-Q kernels
* Demonstrate phase kickback with a controlled unitary and explain its role in QPE
* Build a canonical Quantum Phase Estimation circuit to estimate molecular ground state energies
* Implement Iterative QPE (IQPE) using classical feed-forward logic to reduce qubit requirements
* Implement Bayesian QPE (BQPE) and parallelize experiments using CUDA-Q's MQPU backend
* Compare the tradeoffs between canonical, iterative, and Bayesian QPE approaches

**Prerequisites:**
* Python and Jupyter familiarity
* Basic knowledge of quantum computing (qubits, gates, measurement) — see the [Quick Start to Quantum](https://github.com/NVIDIA/cuda-q-academic/blob/main/quick-start-to-quantum/01_quick_start_to_quantum.ipynb) series
* Familiarity with complex numbers and matrix exponentiation
* Understanding of eigenvalues and eigenstates of Hermitian operators

**Key Terminology:**
* QPE (Quantum Phase Estimation)
* QFT (Quantum Fourier Transform)
* IQFT (Inverse Quantum Fourier Transform)
* Phase Kickback
* Binary Fraction Approximation
* IQPE (Iterative Quantum Phase Estimation)
* BQPE (Bayesian Quantum Phase Estimation)
* Trotterization

**CUDA-Q Syntax:**
* [`@cudaq.kernel`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.kernel) — defines a quantum kernel function
* [`cudaq.qvector`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.qvector) — allocates a register of qubits
* [`cudaq.qubit`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.qubit) — allocates a single qubit
* [`cudaq.sample`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.sample) — samples measurement outcomes
* [`cudaq.control`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.control) — applies a kernel as a controlled operation
* [`cudaq.adjoint`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.adjoint) — applies the adjoint (dagger) of a kernel
* [`cudaq.set_target`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.set_target) — selects simulation or hardware backend
* [`cudaq.run_async`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.run_async) — asynchronous kernel execution (MQPU)
* [`cudaq_solvers.create_molecule`](https://nvidia.github.io/cuda-quantum/latest/api/solvers/python_api.html#cudaq_solvers.create_molecule) — builds molecular Hamiltonian from geometry

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 12px 15px 12px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900;">&#9889; GPU Required:</span>** Parts of this notebook require a GPU.

</div>

In [ ]:
## Instructions for Google Colab. You can ignore this cell if you have CUDA-Q
## set up locally with all required files on your system.
## Uncomment the lines below and execute this cell to install CUDA-Q.

#!pip install cudaq -q
#
#!wget -q https://github.com/nvidia/cuda-q-academic/archive/refs/heads/main.zip
#!unzip -q main.zip
#!mv cuda-q-academic-main/chemistry-simulations/Images ./Images
#!mv cuda-q-academic-main/chemistry-simulations/aux_files ./aux_files

> **Note:** Run the cell below to import all required packages.
> If you installed packages above, restart the kernel first
> (**Runtime → Restart session** in Colab, or **Kernel → Restart** in Jupyter).

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '..'))
sys.path.append(os.path.join(os.getcwd(), '..', 'aux_files', 'qpe'))

import numpy as np
import matplotlib.pyplot as plt

import cudaq
from cudaq import spin

## To install cudaq-solvers (if not already installed), uncomment and run:
## !pip install cudaq-solvers -q
## Note: cudaq-solvers requires libgfortran. If you see an ImportError, run:
## !apt-get install -y libgfortran5
import cudaq_solvers as solvers

from bqpe_plot import plot_bqpe_comparison

---

## 1. Prerequisite Concepts

Before exploring the canonical **QPE (Quantum Phase Estimation)** algorithm, it is useful to explore a few prerequisites which we will use in the algorithm later. These topics will seem very disjointed without context, but understanding each on its own provides a much stronger foundation for understanding what QPE is doing. If you are familiar with one or both of these topics, feel free to skip ahead.


### The Quantum Fourier Transform

The **quantum Fourier transform (QFT)** is a technique inspired by the classical discrete Fourier transform and specifies a particular change of basis following the formula below.

$$\text{QFT} \ket{x} = \frac{1}{\sqrt{N}} \sum_{y=0}^{N-1} e^{\frac{2\pi i xy}{N}} \ket{y}$$

Which results in the following unitary operation.

$$
F_N = \frac{1}{\sqrt{N}}
\begin{bmatrix}
1 & 1 & 1 & \dots & 1 \\
1 & \omega & \omega^2 & \dots & \omega^{N-1} \\
1 & \omega^2 & \omega^4 & \dots & \omega^{2(N-1)} \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & \omega^{N-1} & \omega^{2(N-1)} & \dots & \omega^{(N-1)(N-1)}
\end{bmatrix}
$$

The transformation is generally used to take a state encoded as an integer (e.g. $\ket{3}$ encoded as $\ket{011}$) and encode the integer 3 within the phase information of the state. What this means is that we take a state where one would measure 3 with certainty, perform QFT, and result in a state where we would measure all possible bitstrings with equal probability. So, the information seems lost, but is encoded in the phase which cannot be probed by measurement.

$$
\underbrace{
\frac{1}{\sqrt{8}}
\begin{bmatrix}
1 & 1 & 1 & 1 & 1 & 1 & 1 & 1 \\
1 & \omega & \omega^2 & \omega^3 & \omega^4 & \omega^5 & \omega^6 & \omega^7 \\
1 & \omega^2 & \omega^4 & \omega^6 & 1 & \omega^2 & \omega^4 & \omega^6 \\
1 & \omega^3 & \omega^6 & \omega^1 & \omega^4 & \omega^7 & \omega^2 & \omega^5 \\
1 & \omega^4 & 1 & \omega^4 & 1 & \omega^4 & 1 & \omega^4 \\
1 & \omega^5 & \omega^2 & \omega^7 & \omega^4 & \omega^1 & \omega^6 & \omega^3 \\
1 & \omega^6 & \omega^4 & \omega^2 & 1 & \omega^6 & \omega^4 & \omega^2 \\
1 & \omega^7 & \omega^6 & \omega^5 & \omega^4 & \omega^3 & \omega^2 & \omega^1
\end{bmatrix}
}_{\text{QFT Matrix (Mod 8)}}
\cdot
\underbrace{
\begin{bmatrix}
0 \\ 0 \\ 0 \\ \mathbf{1} \\ 0 \\ 0 \\ 0 \\ 0
\end{bmatrix}
}_{\ket{3}}
=
\frac{1}{\sqrt{8}}
\begin{bmatrix}
1 \\ \omega^3 \\ \omega^6 \\ \omega^1 \\ \omega^4 \\ \omega^7 \\ \omega^2 \\ \omega^5
\end{bmatrix}
\begin{array}{l}
\leftarrow \ket{0} \\
\leftarrow \ket{1} \\
\leftarrow \ket{2} \\
\leftarrow \ket{3} \\
\leftarrow \ket{4} \\
\leftarrow \ket{5} \\
\leftarrow \ket{6} \\
\leftarrow \ket{7}
\end{array}
$$

The process can be reversed using the **inverse QFT (IQFT)** which takes information encoded in a phase, and translates it to a bitstring. Unsurprisingly the formula is

$$\text{QFT}^{\dagger} \ket{y} = \frac{1}{\sqrt{N}} \sum_{x=0}^{N-1} e^{-\frac{2\pi i xy}{N}} \ket{x}$$

And the matrix is just the complex conjugate of the QFT matrix noting that $\omega^{\dagger} = \omega ^{-1}$. The quantum circuits for each of these are also quite easy to implement which you will do in the following exercise.

Try the interactive [QFT widget](https://nvidia.github.io/cuda-q-academic/interactive_widgets/qft.html) to visualize how the QFT transforms basis states into phase-encoded superpositions.


<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 1:</span>**

Use CUDA-Q to code up a kernel for QFT and a kernel for IQFT. Prepare the 3 state on a blank register (you may want to use something like `reg = cudaq.qvector(np.array([0,0,0,1+0j,0,0,0,0]))`) and measure it to confirm 3 comes out 100% of the time. Then apply QFT. Measure the state and confirm you get a mixture of all basis states. Apply IQFT and confirm you get 3 again. As this is a common algorithm, it is a great place to use an AI programming assistant to speed this up.

Next, prepare the three state, but perform a small $R_Y$ rotation on any qubit. What happens to the result after IQFT is performed?

</div>

In [ ]:
# EXERCISE 1

@cudaq.kernel
def quantum_fourier_transform(qubits: cudaq.qview):
    qubit_count = len(qubits)
    
    for i in range(qubit_count):
        h(qubits[i])
        for j in range(i + 1, qubit_count):
            angle = (2 * np.pi) / (2**(j - i + 1))
            r1.ctrl(angle, qubits[j], qubits[i])
    for i in range(qubit_count // 2):
        swap(qubits[i], qubits[qubit_count - 1 - i])

@cudaq.kernel
def inverse_qft(qubits: cudaq.qview):
    cudaq.adjoint(quantum_fourier_transform, qubits)


@cudaq.kernel
def test():
    reg = cudaq.qvector(np.array([0,0,0,1+0j,0,0,0,0]))
    
    quantum_fourier_transform(reg)
    
    ry(np.pi/8, reg[1])

    inverse_qft(reg)

print(cudaq.sample(test))

---

## 2. Phase Kickback

Another important ingredient for understanding QPE is **phase kickback**. First, let's look at an example and then define the phenomenon. Consider the following state:


$$
\frac{\ket{0} + \ket{1}}{\sqrt{2}} \otimes \ket{\psi}
$$

This can be expanded as:

$$
\frac{1}{\sqrt{2}} \left( \ket{0}\ket{\psi} + \ket{1}\ket{\psi} \right)
$$

Now consider a unitary $U$ for which $\psi$ is an eigenstate. If $U$ is applied as an operation controlled by the first qubit the following is obtained.

$$
\frac{1}{\sqrt{2}} \left( \ket{0} (\mathbb{I}\ket{\psi}) + \ket{1} (U\ket{\psi}) \right)
$$

Since $\ket{\psi}$ is an eigenstate of $U$, we substitute $U\ket{\psi} = e^{i\theta}\ket{\psi}$:

$$
\frac{1}{\sqrt{2}} \left( \ket{0}\ket{\psi} + \ket{1} (e^{i\theta}\ket{\psi}) \right)
$$

Because $\ket{\psi}$ is common to both terms, we can factor it out to the right. The phase factor $e^{i\theta}$ is now associated exclusively with the control qubit's $\ket{1}$ state.

$$
 \left( \frac{\ket{0} + e^{i\theta}\ket{1}}{\sqrt{2}} \right) \otimes \ket{\psi}
$$

Something very interesting happened: even though the first qubit did not have anything operate on it, it obtained a phase that was kicked back when the $U$ operated on $\ket{\psi}$. This cannot happen in classical computation where a controlled operation is one way.


<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 2:</span>**

Start with the state below and apply a controlled $Z$ operation where qubit 0 is the control qubit. Recall that $\ket{1}$ is an eigenstate of $Z$ with eigenvalue $e^{i\pi} = -1$. Write a CUDA-Q kernel to prove that phase kickback occurred.

$$\ket{+} \otimes \ket{1} = \left( \frac{\ket{0} + \ket{1}}{\sqrt{2}} \right) \ket{1}$$

</div>

In [ ]:
# EXERCISE 2

@cudaq.kernel
def kickback() -> int:

    reg = cudaq.qvector(2)

    h(reg[0])
    x(reg[1])

    z.ctrl(reg[0], reg[1])

    h(reg[0])

    qubit0 = mz(reg[0])
    
    return qubit0 # returns 1 indicating phase was kicked back

print(cudaq.run(kickback))

---

## 3. Canonical QPE

If you completed the prerequisite steps above, you now have the tools to understand QPE. The goal of QPE in this context is to obtain the energy of a molecular ground state given its Hamiltonian. A key point to note is that the ground state of $H$ is an eigenstate. Before going into the details, let's give a conceptual overview of the algorithm.

1.  Build a register containing the quantum state and a register of some number of ancilla qubits.
2.  Apply controlled operations of a Unitary $U$ (derived from the Hamiltonian $H$; in our implementation $U = e^{iHt}$ via `exp_pauli`) on the state register to encode the energy within the relative phases of the ancilla qubits.
3.  Extract the energy from the phase using the Inverse Quantum Fourier Transform (IQFT).

<img src="../Images/qpe/QPE.png" alt="Diagram showing the three conceptual steps of QPE: state preparation of ancilla and system registers, controlled unitary encoding of eigenphases, and IQFT extraction of the energy as a binary fraction" width="800">

We will now dive into the details of each step, giving you enough information to implement QPE in CUDA-Q on your own.

### State Preparation

Step 1 may seem rather straightforward, but it is actually the hardest part of the algorithm because it is extremely hard to prepare the exact ground state of $H$. (See the Notebook on VQE for ground state preparation). Usually, the state register is prepared as an easily obtained approximation like a Hartree-Fock state or a VQE-optimized state. There are many more sophisticated approaches beyond the scope of this notebook, but some alternatives might be as or more expensive than performing QPE itself, so this remains a serious barrier for implementation. To learn more about one such state preparation method, the Generative Quantum Eigensolver, see the [VQE and GQE lesson](https://github.com/NVIDIA/cuda-q-academic/blob/main/chemistry-simulations/vqe_and_gqe.ipynb) in the CUDA-Q Academic repository.

The probability of measuring the correct ground state energy is proportional to the squared overlap of the prepared state and the true ground state. This means a close approximation might be good enough, but a poor approximation might yield catastrophic results (measuring the wrong eigenenergy).

The ancilla register is much easier to prepare and consists of $n$ qubits, each acted on by a Hadamard gate. The selection of $n$ is arbitrary but controls the precision of the **binary fraction approximation** of the energy. We will circle back to this later, but note that while more ancilla qubits provide a more precise result, each additional ancilla exponentially increases the circuit depth.


### Encoding the Energy on the Phase

The encoding process is pretty simple as depicted in the purple region of the circuit diagram below.

<img src="../Images/qpe/QPE.png" alt="Quantum circuit for canonical QPE showing ancilla qubits initialized with Hadamard gates, controlled-U operations of increasing powers applied to the state register, and the inverse QFT applied to the ancilla register before measurement" width="800">

The first ancilla performs a controlled operation of $U^{2^{n-1}}$ on the state register, the second $U^{2^{n-2}}$, all the way to the final which performs a controlled $U$. Recall applying $U^2$ is the same as applying $U$ twice. So the cost of adding another ancilla qubit is another $2^n$ applications of $U$. This is why QPE is a fault tolerant method that requires exponentially deep circuits.

Each application of $U$ is a time evolution of the system and results in a phase rotation of $e^{i2\pi\theta}$. Because we know that $\ket{\psi}$ is an eigenstate of $H$ we can set this equal to $e^{iEt}$ (since our $U = e^{iHt}$). Resulting in:

$$E = \frac{2\pi\theta}{t}$$

If we could magically read a phase right from a quantum state, we could simply apply a controlled $U$ on the state and look at the accumulated phase on the ancilla qubit. Sadly, we cannot do such a thing as we can only measure 1's and 0's from the QPU and not a decimal like 0.7321, directly.

However, there is a way to extract a decimal from the QPU if we represent it as a binary approximation. The cascade of controlled $U^{2^k}$ operators encodes a single bit of this approximation in the phase of each ancilla. The more ancilla qubits we use, the more precise the binary fraction approximation becomes. Try the interactive [binary fraction widget](https://nvidia.github.io/cuda-q-academic/interactive_widgets/binary_fraction.html) to build intuition for how decimal values are approximated as binary fractions.

The time evolution $U = e^{iHt}$ is implemented via **Trotterization**, which decomposes the operator into a product of simpler exponentials (`exp_pauli` in CUDA-Q) that can each be applied as a gate.

Now where does the $2^k$ factor come from? Consider the approximate form of $\theta$ below.

$$\theta = 0.\phi_1 \phi_2 \phi_3 \dots = \frac{\phi_1}{2^1} + \frac{\phi_2}{2^2} + \frac{\phi_3}{2^3} + \dots$$

The state of the control qubit that applies $U^{2^k}$ becomes

$$\frac{1}{\sqrt{2}}(\ket{0} + e^{2\pi i (2^k \theta)}\ket{1})$$

This essentially multiplies theta by 2, 4, 8, ... and each time moves the decimal in the binary decimal approximation. (Convince yourself that multiplying by 2 does so.) For example:

$$2^2\theta = 4\theta = \phi_1 \phi_2. \phi_3 \dots$$

The integer part $\phi_1\phi_2$ just becomes a factor of $2\pi$ which results in a phase factor of 1. This procedure essentially allows each ancilla to encode a phase corresponding to the most significant digit in the decimal approximation. We just need a tool to extract this information when we measure the ancilla.

### Measuring the Phase

Thankfully we have such a tool which you already coded, the IQFT! Essentially, we have some integer $j$ encoded in the phases of the ancilla qubits. This $j$ is special, because its binary representation is the binary decimal approximation of $\theta$. So, we simply apply IQFT, measure the ancillas and compute theta. The measured bitstring is **MSB … LSB** (left to right): the first ancilla, which had $U^{2^{n-1}}$, gives the most significant bit of $\theta$, and the last ancilla gives the least significant bit.



The formula $E = \frac{2\pi\theta}{t}$ is then used to compute $E$. The $t$ term is the time step for the evolution, this is important to ensure the energy is within the domain of 0 to 1. If not, we can have phase wrapping occur as each factor of $2\pi$ will create an identical phase. For example, rather than try to estimate a phase of 7.5, we could consider a small time step and instead estimate 0.75 which is between 0 and 1.



<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 3:</span>**

Complete the parts of the code below labeled `##TODO##` to build a CUDA-Q implementation of QPE. The molecular preprocessing is completed using CUDA-Q Solvers which produces the Hamiltonian for you and is applied with the provided `controlled_H` function. Notice that the identity term is removed and added back for the final energy. You will:

1. Complete the main QPE CUDA-Q kernel
2. Complete the function that takes the QPE bitstring and computes the energy

</div>

In [ ]:
# EXERCISE 3

cudaq.set_target('nvidia', option='fp64')

geometry = [('H', (0.0, 0.0, 0.0)), ('H', (0.0, 0.0, 0.74))]
molecule = solvers.create_molecule(geometry,
                                   'sto-3g',
                                   0,
                                   0, 
                                   casci=True)

n_qubits  = molecule.n_orbitals*2

print(molecule.hamiltonian)
print(molecule.energies)
coeffs = []
words = []

for term in molecule.hamiltonian:
    coeffs.append(term.evaluate_coefficient().real)
    words.append(term.get_pauli_word(n_qubits))

print(coeffs)
print(words)

identity_coeff = coeffs[0] 

coeffs=coeffs[1:]
words=words[1:]

print(coeffs)
print(words)

import numpy as np

# Default Trotter/time-step parameters (change these to alter all QPE examples consistently)
DEFAULT_TIME_STEP = 0.1
DEFAULT_TROTTER_ORDER = 1

@cudaq.kernel
def controlled_H(state: cudaq.qvector, coeffs: list[float], words: list[cudaq.pauli_word], exponent: int, time_step: float, trotter_order: int):
    x = 0
    while x < exponent:
        for trotter_step in range(trotter_order):
            for i in range(len(coeffs)):
                exp_pauli(coeffs[i]/trotter_order*time_step, state, words[i])
        x+=1


@cudaq.kernel
def quantum_fourier_transform(qubits: cudaq.qview):
    qubit_count = len(qubits)
    
    for i in range(qubit_count):
        h(qubits[i])
        for j in range(i + 1, qubit_count):
            angle = (2 * np.pi) / (2**(j - i + 1))
            r1.ctrl(angle, qubits[j], qubits[i])
    for i in range(qubit_count // 2):
        swap(qubits[i], qubits[qubit_count - 1 - i])

@cudaq.kernel
def inverse_qft(qubits: cudaq.qview):
    cudaq.adjoint(quantum_fourier_transform, qubits)




        

@cudaq.kernel
def canonical_qpe(n_qubits_state: int, n_qubits_ancilla: int, coeffs: list[float], words: list[cudaq.pauli_word], time_step: float, trotter_order: int):
    ancilla = cudaq.qvector(n_qubits_ancilla)
    state = cudaq.qvector(n_qubits_state)
    
    ##TODO## Complete the QPE kernel

    h(ancilla)

    # Prepares hartree Fock state, i.e., lowest two orbitals occupied (1) for H2  |1100>
    x(state[0])
    x(state[1])

    for qubit in range(n_qubits_ancilla):
         cudaq.control(controlled_H, ancilla[n_qubits_ancilla - qubit - 1], state, coeffs, words, 2** qubit, time_step, trotter_order)

    inverse_qft(ancilla)


n_ancilla = 8
time_step = DEFAULT_TIME_STEP
trotter_order = DEFAULT_TROTTER_ORDER

result = cudaq.sample(canonical_qpe, n_qubits, n_ancilla, coeffs, words, time_step, trotter_order, shots_count =10000)

print(result)


def get_energy(result, n_ancilla, time_step=0.1):
    
    bitstring = result.most_probable()
    if len(bitstring) > n_ancilla:
        bitstring = bitstring[:n_ancilla]

        
    ##TODO## Complete the get_energy function

    decimal = int(bitstring, 2)
    phase = decimal / (2**n_ancilla)

    if phase > 0.5:
        phase -= 1

    # Energy E = (2 * pi * phase) / t
    energy = (2 * np.pi * phase) / time_step
    return energy

phase_energy = get_energy(result, n_ancilla, time_step)
total_energy = phase_energy + identity_coeff

bitstring = result.most_probable()
if len(bitstring) > n_ancilla:
    bitstring = bitstring[:n_ancilla]
phase = int(bitstring, 2) / (2**n_ancilla)
if phase > 0.5:
    phase -= 1.0

fci_energy = molecule.energies["fci_energy"]

print("=" * 58)
print("  Canonical QPE results  (time_step = {:.2f})".format(time_step))
print("=" * 58)
print("  Most probable bitstring:   {}".format(bitstring))
print("  Predicted phase θ:         {:+.6f}  (in (-0.5, 0.5])".format(phase))
print()
print("  Energy (E_phase from θ, then add identity):")
print("    E_phase                   {:+.6f}  Ha".format(phase_energy))
print("  + identity (constant)       {:+.6f}  Ha".format(identity_coeff))
print("  " + "-" * 44)
print("  = E_total (QPE)             {:+.6f}  Ha".format(total_energy))
print()
print("  Reference:")
print("    FCI energy                {:+.6f}  Ha".format(fci_energy))
print("    Error vs FCI              {:+.6f}  Ha".format(total_energy - fci_energy))
print("=" * 58)





<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 4:</span>**

Run your QPE code for a range of ancilla values ranging from 6 to 15. The code below will plot the results. Is the overall trend what you would expect? You may notice there are instances where increasing the ancilla count has no impact on the energy. Can you think of an explanation for why this might be?

</div>

In [ ]:
# EXERCISE 4


shots = 10000
ancilla_range = list(range(6, 15))

energies = []
for anc in ancilla_range:
    result_anc = cudaq.sample(canonical_qpe, n_qubits, anc, coeffs, words, time_step, trotter_order, shots_count=shots)
    bs = result_anc.most_probable()
    e = get_energy(result_anc, anc, time_step) + identity_coeff
    energies.append(e)
    # Diagnostics: bitstring length must equal anc; phase from bs is int(bs[:anc],2)/2**anc
    phase_rad = int(bs[:anc], 2) / (2**anc) if len(bs) >= anc else int(bs, 2) / (2**anc)
    if phase_rad > 0.5:
        phase_rad -= 1.0
    print(f"  anc={anc}  len(bs)={len(bs)}  bs={bs!r}  phase={phase_rad:.6f}  E={e:.6f} Ha")


# Diagnostics above: for each anc, check len(bs)==anc (if not, get_energy now uses first anc bits).
# Energies can still match across n when the *phase* aligns: e.g. phase 0.5 = 32/64 (6 anc) = 512/1024 (10 anc).
# If you see identical phases for 6,7,8,9 that is plausible; if len(bs) were wrong we now fix or raise.


def plot_energy_convergence(ancilla_range, energies, exact_energy=None):
    """
    Plots |E - E_FCI| vs ancilla count so convergence is visible even when
    the energy curve is flat (same phase bin for different t).
    """
    nvidia_dark = "#1A1A1A"
    nvidia_mid = "#2A2A2A"
    nvidia_green = "#76b900"
    nvidia_accent = "#444444"
    nvidia_light = "#E0E0E0"
    nvidia_gray = "#AAAAAA"

    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    fig.patch.set_facecolor(nvidia_dark)
    ax.set_facecolor(nvidia_mid)
    ax.tick_params(colors=nvidia_light)
    ax.xaxis.label.set_color(nvidia_light)
    ax.yaxis.label.set_color(nvidia_light)
    ax.title.set_color(nvidia_green)
    for spine in ax.spines.values():
        spine.set_color(nvidia_accent)
    ax.grid(True, ls="-", alpha=0.3, color=nvidia_accent)
    ax.yaxis.set_major_formatter(plt.ScalarFormatter(useOffset=False))

    if exact_energy is not None:
        errors = np.maximum(np.abs(np.array(energies) - exact_energy), 1e-12)
        ax.semilogy(ancilla_range, errors, 's-', color=nvidia_green, linewidth=2,
                    markersize=8, label='|E_QPE − E_FCI|')
        ax.set_ylabel('|E_QPE − E_FCI| (Ha)', fontsize=12)
        ax.set_title('Error vs FCI (convergence)', fontsize=14)
        ax.legend(frameon=True, facecolor=nvidia_mid, edgecolor=nvidia_accent,
                  labelcolor=nvidia_light, fontsize=10)
    ax.set_xlabel('Number of ancilla qubits (t)', fontsize=12)
    plt.tight_layout()
    plt.show()


plot_energy_convergence(ancilla_range, energies, fci_energy)


---

## 4. Iterative QPE

The deep circuits required for canonical QPE are far too deep for running on anything but a fault-tolerant QPU. With this in mind, other variants of QPE have been developed which allow for tradeoffs in how the phase estimation is performed.

One example is **iterative QPE (IQPE)** where the $n$ ancilla qubits and full deep circuit is traded off for $n$ circuit runs which connect using feed-forward logic. This allows for runs on devices with fewer qubits. The idea is to run a series of circuits like the two in the image below.

<img src="../Images/qpe/IQPE.png" alt="Two circuits illustrating iterative QPE: each uses a single ancilla qubit with a controlled-U operation of decreasing power, an Rz correction gate based on previously measured bits, and measurement to extract one bit of the phase estimate" width="1200">

After state prep, the unitary with the highest power is applied as a controlled operation and it kicks back a phase which is measured. (In the first round $\beta=0$). This provides the LSB of the final phase approximation. Then, we repeat with the next unitary, but this time we apply a $R_Z(\beta)$ gate where beta is calculated as the currently known phase times $-2\pi$. This essentially resets the phase, so each circuit only measures the contribution of the particular unitary.


For example, where we would have $n$ ancilla with canonical QPE, we now have $n$ circuits to run. Say we compute a measurement of 1 in the first and 0 in the second run. Then, the $\beta$ for the third circuit is:

$$\beta_{n-2} = -2 \pi \left(\frac{1}{2^n} + \frac{0}{2^{n-1}} \right)$$

Often, a single shot is not used, but a majority vote from which we determine if the result is a 0 or a 1 to feed-forward to the next circuit.

There is clearly a space savings, as we only require a single ancilla qubit. However, we still need to apply each circuit sequentially, and we still have to run a circuit as deep as the highest power controlled unitary requires.


<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 5:</span>**

Fix the code below where it says `##TODO##` to complete the IQPE code. The kernel is quite similar, so your job is to write the loop for the iterative steps. Confirm that you can reproduce the same result as canonical QPE. Note that the bit ordering for canonical QPE and IQPE is reversed, so we reverse when computing the phase at the end.

</div>

In [ ]:
# EXERCISE 5

n_bits = 8
shots_per_round = 2000  
time_step = DEFAULT_TIME_STEP
trotter_order = DEFAULT_TROTTER_ORDER

# Kernel for one round: single ancilla + state register.
# sample() measures the full register; we extract the ancilla (qubit 0) in post-processing.
@cudaq.kernel
def iterative_qpe_round(
    n_qubits_state: int,
    coeffs: list[float],
    words: list[cudaq.pauli_word],
    exponent: int,
    feedback_angle: float,
    time_step: float,
    trotter_order: int,
):
    ancilla = cudaq.qubit()
    state = cudaq.qvector(n_qubits_state)

    x(state[0])
    x(state[1])
    h(ancilla)
    cudaq.control(controlled_H, ancilla, state, coeffs, words, exponent, time_step, trotter_order)
    rz(feedback_angle, ancilla)
    h(ancilla)

# Run n_bits rounds; each round gives one bit (LSB first: round 0 with k=2^(n-1) -> bits[0], ..., MSB last -> bits[n_bits-1]).

##TODO## - Write the loop for IQPE and perform post-processing to extract the phase. 
bits = []
for j in range(n_bits):
    k = 2 ** (n_bits - 1 - j)
    # Known phase so far from prior bits (LSB-first): 0.b0 b1 ... b(j-1)
    known_phase = sum(bits[i] * (2.0 ** (-(i + 1))) for i in range(j))
    feedback_angle = -2.0 * np.pi * known_phase
    result = cudaq.sample(
        iterative_qpe_round,
        n_qubits,
        coeffs,
        words,
        k,
        feedback_angle,
        time_step,
        trotter_order,
        shots_count=shots_per_round,
    )
    # Ancilla is qubit 0 (first allocated), so it is the first character of each bitstring.
    count0, count1 = 0, 0
    for bitstring in result:
        count = result[bitstring]
        if bitstring[0] == "0":
            count0 += count
        else:
            count1 += count
    bit = 1 if count1 > count0 else 0
    bits.append(bit)

# We collected LSB first (round 0 -> bits[0]), ... MSB last (bits[n_bits-1]).
# Canonical-style bitstring (MSB...LSB) is reversed(bits), so phase = int(reversed(bits))/2^n:
phase = sum(bits[i] * (2.0 ** (-(n_bits - i))) for i in range(n_bits))
if phase > 0.5:
    phase -= 1.0
energy_iqpe = (2.0 * np.pi * phase) / time_step
total_energy_iqpe = energy_iqpe + identity_coeff

# Expected from canonical (bitstring 11111100): bits = [0,0,1,1,1,1,1,1], total ~ -1.08 hartree
print("Iterative QPE (n=8) bits [LSB ... MSB]:", bits)
print("Canonical-style bitstring (MSB...LSB):", "".join(str(b) for b in reversed(bits)))
print("Phase (in [0,1), then wrapped):", phase)
print("Hamiltonian energy (hartree):", energy_iqpe)
print("Total energy (hartree):", total_energy_iqpe)
print("FCI (exact, hartree):", molecule.energies["fci_energy"])
print("(Canonical n=8 gave ~ -1.078 hartree; use more shots_per_round if iterative disagrees.)")

---

## 5. Bayesian QPE

Though IQPE helps, it is still quite susceptible to noise with its relatively deep circuits and is not much faster given the sequential nature of the controlled unitary operations.

**Bayesian QPE (BQPE)** provides a means to address both of these issues and convert QPE into a shot-based statistical approach that is more resilient to noise. The BQPE workflow is shown below.


<img src="../Images/qpe/BQPE.png" alt="Flowchart of the Bayesian QPE workflow: initialize a uniform prior distribution, run shallow circuits with random k and beta parameters, update the posterior distribution using Bayes theorem after each measurement, and repeat until the posterior converges to a sharp peak at the estimated phase" width="1200">


First, a prior distribution $p(\theta)$ is initialized as a uniform distribution over 0 to $2\pi$ as we know nothing about the phase *a priori*. Next, the same circuit we ran for IQPE is run for random pairs of $k$ and $\beta$. The $k$ term still controls the level of precision in the phase estimate measured by that circuit. The $\beta$ parameter is no longer a set parameter and instead randomly selected.

The individual measurements of these "experiments" inform updates to form the posterior $p(\theta \mid m)$ by multiplying the prior by the likelihood $p(m \mid \theta, k, \beta)$ according to Bayes' theorem:

$$p(\theta \mid m) \propto p(m \mid \theta, k, \beta) \times p(\theta)$$

The likelihood is defined as

$$p(m \mid \theta, k, \beta) = \frac{1}{2}(1 + (-1)^m \cdot \cos(\pi \cdot (k \cdot \phi + \beta)))$$

where $m$ is a measurement result of 0 or 1. After each update, the resulting posterior becomes the prior for the next experiment.

Because the approach is statistical, the result is more resilient to noise as it is just part of the statistical estimation of the phase. The challenge for BQPE becomes the cost of performing sufficiently many measurements and storing all of the resulting coefficients necessary to build the most recent posterior distribution which generally grows at least quadratically given the number of circuit runs.

However, BQPE is also completely parallelizable. Every experiment can run on a different QPU as they are independent. CUDA-Q's MQPU backend is well suited for this sort of application as the code can be written to execute on multiple GPU-simulated QPUs at the same time using the `cudaq.run_async()` API.

The `run` API is perfect for running single shots where we want the kernel to return a value, in this case, the measurement of the single ancilla. With IQPE, we ran the same circuit many times, so it was more efficient to use `sample` and postprocess the ancilla measurement.

The `async` capability allows us to run all of the experiments, and save the results as a list of futures which can be accessed with `get`. For more info on this, see the CUDA-Q docs. If you have more than one GPU, you can specify a `qpu_id` parameter which will decide which simulated QPU the command is executed on.


<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 6:</span>**

Fix the code below where it says `##TODO##` to complete the BQPE code. When you are finished, run two experiments where you use 500 and then 3000 samples. Comment on the results. Does the number of samples make a difference?

</div>

In [ ]:
# EXERCISE 6

# ---- Set MQPU Backend ----
cudaq.set_target("nvidia", option="mqpu")
target = cudaq.get_target()
n_qpus = target.num_qpus()
    
print("Using MQPU backend: {} QPU(s) available.".format(n_qpus))

# ---- Use same Trotter/time-step defaults as other QPE examples ----
time_step = DEFAULT_TIME_STEP
trotter_order = DEFAULT_TROTTER_ORDER

# ---- Defined BQPE Circuit w/ return type for cudaq.run ----
@cudaq.kernel
def bqpe_single_shot(k: int, rz_angle: float, n_qubits_state: int, coeffs: list[float], words: list[cudaq.pauli_word], time_step: float, trotter_order: int) -> int:
    ancilla = cudaq.qubit()
    state = cudaq.qvector(n_qubits_state)
    
    x(state[0])
    x(state[1])
    
    h(ancilla)
    cudaq.control(controlled_H, ancilla, state, coeffs, words, k, time_step, trotter_order)
    rz(rz_angle, ancilla)
    h(ancilla)
    
    measurement = mz(ancilla)
    
    return measurement

# ---- Define initial Prior (use same time_step for energy conversion) ----
DELTAT = time_step
n_grid = 2**12 
phi_grid = np.linspace(-1, 1, n_grid + 1)[:-1]
dphi = 2.0 / n_grid
prior = np.ones(n_grid) / (2.0)



# ---- Define Likelihood ----
def likelihood(m, k, beta, phi):
    return 0.5 * (1 + ((-1) ** m) * np.cos(np.pi * (k * phi + beta)))

# ---- Define Posterior Update (Normalized)----
def update_posterior(prior, phi, k, beta, m):
    ##TODO## - Write the update function for the posterior. 
    L = likelihood(m, k, beta, phi)
    post = prior * np.maximum(L, 1e-20)
    
    # Normalize the posterior
    post /= np.sum(post) * dphi
    return post

# ---- Mean and uncertainty from posterior ----
def posterior_to_mean_sigma(post, phi):
    mu_phi = np.sum(phi * post) * dphi
    sigma_phi = np.sqrt(np.sum(post * (phi - mu_phi) ** 2) * dphi)
    return mu_phi, sigma_phi


# ---- Run BQPE Experiment ----

def run_bqpe_experiment_mqpu(n_samples: int, k_max: int):
    
    # Random (k, β): k in 1..k_max, β in [-1, 1)
    k_list = np.random.randint(1, k_max + 1, size=n_samples)
    beta_list = (1 - 2 * np.random.random(size=n_samples)).tolist()
    
    # Launch ALL (k,β) experiments in parallel via run_async
    futures = []
    for i in range(n_samples):
        rz_angle = float(np.pi * beta_list[i])
        fut = cudaq.run_async(
            bqpe_single_shot, int(k_list[i]), rz_angle, n_qubits,
            coeffs, words, time_step, trotter_order,
            shots_count=1, qpu_id=i % n_qpus,
        )
        futures.append(fut)
        
    # Aggregate: each .get() returns a list with one int (the single measurement)
    results_raw = [f.get() for f in futures]
    ms = [int(r[0]) for r in results_raw]
    
    # Posterior updates from aggregated (k, β, m) data
    post = np.copy(prior)
    for k, b, m in zip(k_list, beta_list, ms):
        post = update_posterior(post, phi_grid, k, b, m)
    mu_phi, sigma_phi = posterior_to_mean_sigma(post, phi_grid)
    
    
    # exp_pauli implements e^{+iHΔt}, so phase kicked back = +E_eff·Δt·k → θ = E_eff·Δt/π → E_eff = πθ/Δt
    mu_energy = np.pi * mu_phi / DELTAT + identity_coeff
    sigma_energy = np.pi * sigma_phi / DELTAT
    return post, mu_energy, sigma_energy

# Run two experiments in parallel on MQPU, then plot with FCI reference
fci_energy = molecule.energies["fci_energy"]
exp1_post, exp1_mu, exp1_sigma = run_bqpe_experiment_mqpu(500, k_max=13)
exp2_post, exp2_mu, exp2_sigma = run_bqpe_experiment_mqpu(3000, k_max=13)

print("Experiment 1 (500 samples): μ = {:.5f} Ha, σ = {:.5f} Ha".format(exp1_mu, exp1_sigma))
print("Experiment 2 (3000 samples): μ = {:.5f} Ha, σ = {:.5f} Ha".format(exp2_mu, exp2_sigma))
print("FCI reference:            {:.5f} Ha".format(fci_energy))

fig = plot_bqpe_comparison(
    posteriors_list=[[exp1_post], [exp2_post]],
    mu_list=[[exp1_mu], [exp2_mu]],
    sigma_list=[[exp1_sigma], [exp2_sigma]],
    labels=["BQPE 500 samples", "BQPE 3000 samples"],
    fci_energy=fci_energy,
    delta_t=DELTAT,
    identity_coeff=identity_coeff,
    n_grid=n_grid,
    x_range=(-1.45, -0.95),
)
plt.show()

There are many ways to extend all of these methods, but we will mention one which might be interesting for learners who want to see a recent experimental demonstration of BQPE. In 2023 Quantinuum demonstrated the first experimental fault-tolerant implementation of BQPE run on their H2 ion-trap quantum computer ([arXiv:2306.16608](https://arxiv.org/abs/2306.16608)). They used a simplified single qubit representation of the hydrogen molecule and a quantum error detection code which allowed them to detect errors and discard the bad shots before posterior updates are made. This solidifies the fact that of the three methods presented in this notebook, BQPE is the best suited for near-term experimentation.

## Conclusion

After completing this notebook, you should now have better intuition for one of the most fundamental quantum algorithms.  QPE is important for applications beyond chemistry such as its role in [factoring numbers with Shor's algorithm](https://nvidia.github.io/cuda-quantum/latest/applications/python/shors.html). 

You were also able to learn about the different variations of QPE and how they each come with their own tradeoffs.  The table below summarizes this. 

<table style="width:100%; border-collapse:collapse; font-family: Inter, -apple-system, sans-serif; font-size:14px; color:#CCC; background-color:#303030; border-radius:6px; overflow:hidden;">
  <thead>
    <tr style="background-color:#383838; border-bottom:2px solid #76b900;">
      <th style="padding:14px 16px; text-align:left; color:#9ad035; font-size:13px; text-transform:uppercase; letter-spacing:0.5px;"></th>
      <th style="padding:14px 16px; text-align:center; color:#9ad035; font-size:13px; text-transform:uppercase; letter-spacing:0.5px;">Canonical QPE</th>
      <th style="padding:14px 16px; text-align:center; color:#9ad035; font-size:13px; text-transform:uppercase; letter-spacing:0.5px;">Iterative (Kitaev) QPE</th>
      <th style="padding:14px 16px; text-align:center; color:#9ad035; font-size:13px; text-transform:uppercase; letter-spacing:0.5px;">Bayesian QPE (BQPE)</th>
    </tr>
  </thead>
  <tbody>
    <tr style="border-bottom:1px solid #4a4a4a;">
      <td style="padding:12px 16px; font-weight:600; color:#9ad035;">Ancilla qubits</td>
      <td style="padding:12px 16px; text-align:center;">$n$ (one per bit of precision)</td>
      <td style="padding:12px 16px; text-align:center;">1 (reused across $n$ rounds)</td>
      <td style="padding:12px 16px; text-align:center;">1 (reused across all shots)</td>
    </tr>
    <tr style="border-bottom:1px solid #4a4a4a; background-color:#363636;">
      <td style="padding:12px 16px; font-weight:600; color:#9ad035;">Circuit depth</td>
      <td style="padding:12px 16px; text-align:center;">Deep — one long circuit with controlled-$U^{2^k}$ for all $k$ plus IQFT</td>
      <td style="padding:12px 16px; text-align:center;">Moderate — $n$ separate circuits, each with one controlled-$U^{2^j}$</td>
      <td style="padding:12px 16px; text-align:center;">Shallow — each circuit has a single controlled-$U^k$ (random $k$, no IQFT)</td>
    </tr>
    <tr style="border-bottom:1px solid #4a4a4a;">
      <td style="padding:12px 16px; font-weight:600; color:#9ad035;">Number of circuits</td>
      <td style="padding:12px 16px; text-align:center;">1 (run many shots of the same circuit)</td>
      <td style="padding:12px 16px; text-align:center;">$n$ rounds &times; many shots per round (majority vote), e.g.&nbsp;8&nbsp;&times;&nbsp;2000&nbsp;=&nbsp;16,000</td>
      <td style="padding:12px 16px; text-align:center;">Many (hundreds to thousands of short circuits)</td>
    </tr>
    <tr style="border-bottom:1px solid #4a4a4a; background-color:#363636;">
      <td style="padding:12px 16px; font-weight:600; color:#9ad035;">Classical post-processing</td>
      <td style="padding:12px 16px; text-align:center;">Minimal — read bitstring, convert to phase</td>
      <td style="padding:12px 16px; text-align:center;">Light — classical feedback between rounds</td>
      <td style="padding:12px 16px; text-align:center;">Heavy — Bayesian posterior updates on a phase grid after every measurement</td>
    </tr>
    <tr style="border-bottom:1px solid #4a4a4a;">
      <td style="padding:12px 16px; font-weight:600; color:#9ad035;">Precision</td>
      <td style="padding:12px 16px; text-align:center;">Deterministic $n$-bit resolution (set by number of ancillas)</td>
      <td style="padding:12px 16px; text-align:center;">Deterministic $n$-bit resolution (set by number of rounds)</td>
      <td style="padding:12px 16px; text-align:center;">Statistical — improves with more samples; not limited to fixed bit resolution</td>
    </tr>
    <tr style="border-bottom:1px solid #4a4a4a; background-color:#363636;">
      <td style="padding:12px 16px; font-weight:600; color:#9ad035;">Noise resilience</td>
      <td style="padding:12px 16px; text-align:center;">Low — deep circuit accumulates errors; needs full error correction</td>
      <td style="padding:12px 16px; text-align:center;">Moderate — shorter individual circuits, but later rounds still need high-power $U$</td>
      <td style="padding:12px 16px; text-align:center;">High — shallow circuits; naturally compatible with error detection (e.g., Iceberg code)</td>
    </tr>
    <tr style="border-bottom:1px solid #4a4a4a;">
      <td style="padding:12px 16px; font-weight:600; color:#9ad035;">Parallelizable</td>
      <td style="padding:12px 16px; text-align:center;">No — single monolithic circuit</td>
      <td style="padding:12px 16px; text-align:center;">No — rounds are sequential (each depends on the previous bit)</td>
      <td style="padding:12px 16px; text-align:center;">Yes — all $(k, \beta)$ circuits are independent and can run in parallel across multiple QPUs (e.g., CUDA-Q MQPU)</td>
    </tr>
    <tr style="border-bottom:1px solid #4a4a4a; background-color:#363636;">
      <td style="padding:12px 16px; font-weight:600; color:#9ad035;">State preparation</td>
      <td style="padding:12px 16px; text-align:center;" colspan="3">All three methods require a good approximation to the ground state (e.g., Hartree-Fock, VQE). This is the common bottleneck.</td>
    </tr>
    <tr>
      <td style="padding:12px 16px; font-weight:600; color:#9ad035;">Key tradeoff</td>
      <td style="padding:12px 16px; text-align:center;">Fewest total circuit runs, but requires the deepest circuits and most ancilla qubits</td>
      <td style="padding:12px 16px; text-align:center;">Saves qubits at the cost of sequential rounds with mid-circuit measurement and classical feedback</td>
      <td style="padding:12px 16px; text-align:center;">Shallowest circuits, but requires many more runs and non-trivial classical Bayesian inference</td>
    </tr>
  </tbody>
</table>

Though QPE is powerful in theory, the state preparation problem remains a substantial barrier to its benefits. Clever state preparation methods such as those that leverage generative AI will be required to eventually realize the full potential of QPE. To learn more about one such state preparation method, the Generative Quantum Eigensolver, see the [VQE and GQE lesson](https://github.com/NVIDIA/cuda-q-academic/blob/main/chemistry-simulations/vqe_and_gqe.ipynb) in the CUDA-Q Academic repository.

**Related Notebooks:**
* [Ground State Preparation of Molecules Using VQE and GQE](https://github.com/NVIDIA/cuda-q-academic/blob/main/chemistry-simulations/vqe_and_gqe.ipynb) — explores variational ground state methods referenced as state preparation for QPE
* [Krylov Subspace Diagonalization](https://github.com/NVIDIA/cuda-q-academic/blob/main/chemistry-simulations/krylov_subspace_diagonalization.ipynb) — an alternative non-variational approach to molecular energy estimation
* [Quick Start to Quantum 01](https://github.com/NVIDIA/cuda-q-academic/blob/main/quick-start-to-quantum/01_quick_start_to_quantum.ipynb) — introduces foundational CUDA-Q concepts used throughout this notebook